# What predicts student dropout?

The UCI *Predict Students' Dropout and Academic Success* dataset has 4,424 students and 36 features, with a three-class target (Dropout / Enrolled / Graduate). The end goal is a classifier, so this EDA asks one question over and over: **which features separate dropouts from everyone else, and when is that information available?**

The point of the project is *early* intervention. Features fall into two groups:

- **Known at enrollment:** demographics, application path, parents' background, admission grade, finances, and macro-economic conditions.
- **Known only after a semester or two:** the `Curricular units ...` columns.

A model that leans on second-semester grades is accurate but late, so that distinction stays in view throughout.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PALETTE = {"Graduate": "#2a78d6", "Dropout": "#eb6834", "Enrolled": "#1baf7a"}
ORDER = ["Dropout", "Enrolled", "Graduate"]
INK, MUTED = "#0b0b0b", "#52514e"

plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED,
    "ytick.color": MUTED, "axes.grid": True, "grid.color": "#e6e5e1",
    "grid.linewidth": 0.6, "axes.axisbelow": True, "font.size": 10,
})

## Load and clean

The file is semicolon-delimited, and one column name has a stray tab, so whitespace is stripped from every column name.

In [4]:
from competitions.config import RAW_DATA_DIR

df = pd.read_csv(RAW_DATA_DIR / "data.csv", sep=";")
df.columns = df.columns.str.strip()
df.shape

ModuleNotFoundError: No module named 'competitions'

In [ ]:
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
df.dtypes.value_counts()

No missing values and no duplicates. Every categorical feature is stored as an integer code, so readable labels are added for the features used in plots, along with a binary `dropout` flag.

In [ ]:
course_names = {
    33: "Biofuel Production Tech", 171: "Animation & Multimedia", 8014: "Social Service (evening)",
    9003: "Agronomy", 9070: "Communication Design", 9085: "Veterinary Nursing",
    9119: "Informatics Engineering", 9130: "Equinculture", 9147: "Management",
    9238: "Social Service", 9254: "Tourism", 9500: "Nursing", 9556: "Oral Hygiene",
    9670: "Advertising & Marketing", 9773: "Journalism & Communication",
    9853: "Basic Education", 9991: "Management (evening)",
}
app_mode_names = {
    1: "1st phase, general", 17: "2nd phase, general", 18: "3rd phase, general",
    39: "Over 23 years old", 43: "Change of course", 44: "Tech. specialization diploma",
    7: "Holders of other higher courses", 42: "Transfer", 51: "Change of inst./course",
    53: "Short cycle diploma", 15: "International student",
}

df["course_name"] = df["Course"].map(course_names)
df["app_mode_name"] = df["Application mode"].map(app_mode_names).fillna("Other")
df["Target"] = pd.Categorical(df["Target"], categories=ORDER)
df["dropout"] = (df["Target"] == "Dropout").astype(int)

## The target

`Enrolled` is the smallest class and usually sits between the other two. Treating the task as **Dropout vs. not** is a reasonable simplification if the goal is flagging at-risk students.

In [ ]:
counts = df["Target"].value_counts().reindex(ORDER)
fig, ax = plt.subplots(figsize=(6, 2.4))
ax.barh(counts.index, counts.values, color=[PALETTE[k] for k in counts.index], height=0.6)
for i, v in enumerate(counts.values):
    ax.text(v + 30, i, f"{v:,}  ({v / len(df):.0%})", va="center", color=INK)
ax.set_xlim(0, counts.max() * 1.25)
ax.set_title("Outcome at the end of the normal course duration", loc="left")
ax.grid(axis="y", visible=False)
ax.invert_yaxis()
plt.show()

## A data-quality wrinkle: students with zero units

In [ ]:
cu = [c for c in df.columns if c.startswith("Curricular units")]
no_units = df[cu].sum(axis=1).eq(0)
print(f"{no_units.sum()} students have 0 in every curricular-unit column")
df.loc[no_units, "Target"].value_counts()

df["no_units"] = no_units.astype(int)
active = df[~no_units]

These students enrolled in, were credited for, and were evaluated in nothing in either semester, yet some are labelled Graduate. This means `grade == 0` does not always mean failed. Keep a flag, or exclude these rows when looking at grades.

## Features known at enrollment

### Money is the strongest early signal

In [ ]:
def rate_table(col, labels=None):
    t = df.groupby(col, observed=True)["dropout"].agg(rate="mean", n="size")
    if labels:
        t.index = t.index.map(labels)
    return t

binary = {
    "Tuition fees up to date": {0: "Not up to date", 1: "Up to date"},
    "Debtor": {0: "Not a debtor", 1: "Debtor"},
    "Scholarship holder": {0: "No scholarship", 1: "Scholarship"},
    "Gender": {0: "Female", 1: "Male"},
    "Daytime/evening attendance": {0: "Evening", 1: "Daytime"},
    "Displaced": {0: "Not displaced", 1: "Displaced"},
}

fig, axes = plt.subplots(2, 3, figsize=(10, 5), sharex=True)
base = df["dropout"].mean()
for ax, (col, labels) in zip(axes.flat, binary.items()):
    t = rate_table(col, labels)
    ax.barh(t.index, t["rate"], color=PALETTE["Dropout"], height=0.55)
    ax.axvline(base, color=MUTED, ls="--", lw=1)
    for i, (r, n) in enumerate(zip(t["rate"], t["n"])):
        ax.text(r + 0.02, i, f"{r:.0%}  (n={n:,})", va="center", fontsize=9, color=INK)
    ax.set_title(col, loc="left", fontsize=10)
    ax.set_xlim(0, 1.15)
    ax.grid(axis="y", visible=False)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
fig.suptitle("Dropout rate by group (dashed line = overall rate)", x=0.01, ha="left")
fig.tight_layout()
plt.show()

In [ ]:
pd.crosstab(df["Debtor"], df["Tuition fees up to date"], normalize="index").round(2)

### Age and application route

Age is right-skewed with a long tail, so binning or log-transforming it may help a linear model. The "Over 23 years old" application route is closely related to age.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for target in ORDER:
    sns.kdeplot(df.loc[df["Target"] == target, "Age at enrollment"], ax=ax,
                color=PALETTE[target], lw=2, label=target, cut=0, common_norm=False)
ax.set_xlabel("Age at enrollment")
ax.set_ylabel("Density")
ax.set_title("Age distribution by outcome", loc="left")
ax.legend(frameon=False)
plt.show()

age_bins = pd.cut(df["Age at enrollment"], [16, 19, 22, 25, 30, 40, 70],
                  labels=["17-19", "20-22", "23-25", "26-30", "31-40", "41+"])
df.groupby(age_bins, observed=True)["dropout"].agg(rate="mean", n="size").round(3)

In [ ]:
t = (df.groupby("app_mode_name", observed=True)["dropout"].agg(rate="mean", n="size")
       .query("n >= 50").sort_values("rate"))
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(t.index, t["rate"], color=PALETTE["Dropout"], height=0.6)
ax.axvline(base, color=MUTED, ls="--", lw=1)
for i, (r, n) in enumerate(zip(t["rate"], t["n"])):
    ax.text(r + 0.01, i, f"{r:.0%}  (n={n:,})", va="center", fontsize=9, color=INK)
ax.set_xlim(0, 0.8)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.set_title("Dropout rate by application mode (groups with n >= 50)", loc="left")
ax.grid(axis="y", visible=False)
plt.show()

### Course matters a lot

`Course` is a categorical code, not a numeric measurement, so it should be one-hot encoded.

In [ ]:
t = df.groupby("course_name", observed=True)["dropout"].agg(rate="mean", n="size").sort_values("rate")
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(t.index, t["rate"], color=PALETTE["Dropout"], height=0.65)
ax.axvline(base, color=MUTED, ls="--", lw=1)
for i, (r, n) in enumerate(zip(t["rate"], t["n"])):
    ax.text(r + 0.01, i, f"{r:.0%}  (n={n})", va="center", fontsize=9, color=INK)
ax.set_xlim(0, 0.85)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.set_title("Dropout rate by course", loc="left")
ax.grid(axis="y", visible=False)
plt.show()

pd.crosstab(df["course_name"], df["Gender"].map({0: "Female", 1: "Male"}),
            normalize="index").round(2).sort_values("Male", ascending=False).head(6)

### Prior achievement

Admission grades overlap substantially across outcomes, so entry grades alone are unlikely to flag many dropouts.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, col in zip(axes, ["Admission grade", "Previous qualification (grade)"]):
    sns.boxplot(data=df, x="Target", y=col, hue="Target", order=ORDER, palette=PALETTE,
                ax=ax, width=0.5, fliersize=2, legend=False)
    ax.set_title(col, loc="left")
    ax.set_xlabel("")
    ax.set_ylabel("Grade (0-200)")
plt.tight_layout()
plt.show()

df.groupby("Target", observed=True)[["Admission grade", "Previous qualification (grade)"]].median()

## Features known after the first year

### First-semester performance separates the classes sharply

In [ ]:
act = active.copy()
for semester in ["1st", "2nd"]:
    enrolled = act[f"Curricular units {semester} sem (enrolled)"]
    act[f"pass_rate_{semester}"] = act[f"Curricular units {semester} sem (approved)"] / enrolled.replace(0, np.nan)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
sns.kdeplot(data=act, x="pass_rate_1st", hue="Target", hue_order=ORDER, palette=PALETTE,
            common_norm=False, cut=0, lw=2, ax=axes[0])
axes[0].set_title("Share of first-semester units passed", loc="left")
axes[0].set_xlabel("Units approved / units enrolled")
sns.kdeplot(data=act, x="Curricular units 1st sem (grade)", hue="Target", hue_order=ORDER,
            palette=PALETTE, common_norm=False, cut=0, lw=2, ax=axes[1])
axes[1].set_title("First-semester grade average (0-20)", loc="left")
for ax in axes:
    ax.get_legend().set_frame_on(False)
plt.tight_layout()
plt.show()

zero_pass = df["Curricular units 1st sem (approved)"].eq(0) & ~no_units
print(f"Students who passed 0 units in the 1st semester: {zero_pass.sum()}")
df.loc[zero_pass, "Target"].value_counts(normalize=True).round(3)

### Trajectory from semester 1 to semester 2

In [ ]:
act["grade_change"] = (act["Curricular units 2nd sem (grade)"]
                       - act["Curricular units 1st sem (grade)"])
act.groupby("Target", observed=True)[["Curricular units 1st sem (grade)",
                                      "Curricular units 2nd sem (grade)",
                                      "grade_change", "pass_rate_1st",
                                      "pass_rate_2nd"]].median().round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for target in ["Graduate", "Enrolled", "Dropout"]:
    d = act[act["Target"] == target]
    ax.scatter(d["Curricular units 1st sem (grade)"], d["Curricular units 2nd sem (grade)"],
               s=12, alpha=0.35, color=PALETTE[target], label=target, edgecolors="none")
ax.plot([0, 20], [0, 20], color=MUTED, lw=1, ls="--")
ax.set_xlabel("First-semester grade average")
ax.set_ylabel("Second-semester grade average")
ax.set_title("Grades in semester 1 vs semester 2", loc="left")
ax.legend(frameon=False, markerscale=2)
plt.show()

## How strong is each feature overall?

Mutual information captures non-linear and categorical relationships that correlation misses. It can be biased upward for high-cardinality features, so interpret the ranking with group rates and domain knowledge.

In [ ]:
from sklearn.feature_selection import mutual_info_classif

features = [c for c in df.columns
            if c not in ["Target", "dropout", "course_name", "app_mode_name", "no_units"]]
discrete = [df[c].nunique() < 30 or c in ["Course", "Application mode", "Nacionality",
            "Mother's occupation", "Father's occupation", "Mother's qualification",
            "Father's qualification", "Previous qualification"] for c in features]
mi = pd.Series(mutual_info_classif(df[features], df["Target"].cat.codes,
                                   discrete_features=discrete, random_state=0),
               index=features).sort_values()

timing = ["After enrollment" if c.startswith("Curricular") else "At enrollment" for c in mi.index]
colors = ["#86b6ef" if t == "After enrollment" else "#1c5cab" for t in timing]

fig, ax = plt.subplots(figsize=(8, 8))
ax.barh(mi.index, mi.values, color=colors, height=0.7)
ax.set_title("Mutual information with outcome", loc="left")
ax.set_xlabel("Mutual information (higher = more informative)")
ax.grid(axis="y", visible=False)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="#1c5cab", label="Known at enrollment"),
                   Patch(color="#86b6ef", label="Known after 1st/2nd semester")],
          frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

## How early can we predict?

This is a quick, untuned comparison using a gradient-boosted classifier and five-fold cross-validated macro-F1.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

enroll_feats = [c for c in features if not c.startswith("Curricular")]
sem1_feats = enroll_feats + [c for c in features if "1st sem" in c]
all_feats = features

cv = StratifiedKFold(5, shuffle=True, random_state=0)
y = df["Target"].cat.codes
results = {}
for name, cols in [("Enrollment only", enroll_feats),
                   ("+ 1st semester", sem1_feats),
                   ("+ 2nd semester", all_feats)]:
    cat_mask = [c in ["Course", "Application mode", "Marital status", "Nacionality",
                      "Previous qualification"] for c in cols]
    model = HistGradientBoostingClassifier(categorical_features=cat_mask, random_state=0)
    scores = cross_val_score(model, df[cols], y, cv=cv, scoring="f1_macro")
    results[name] = {"macro F1": scores.mean().round(3), "+/- sd": scores.std().round(3)}
pd.DataFrame(results).T

## Takeaways for modeling

1. **Class imbalance:** use stratified splits and macro-F1 or per-class recall, not accuracy alone.
2. **Strongest early signals:** tuition status, debtor status, scholarship, age, application route, gender, attendance timing, and course.
3. **Strongest overall signal:** semester performance, especially passing zero units and a collapse to zero in the second semester.
4. **Weak signals:** admission grade, parents' background, nationality, and macro-economic variables.
5. **Preprocessing:** one-hot encode categorical codes, collapse rare levels where appropriate, and engineer pass rates, zero-approved flags, and grade change.
6. **Use case first:** an early-intervention model should use enrollment features, optionally first-semester features, and exclude second-semester information.